
Remember to have the required environment configured. This relies on devenv-hgq (from environment-HGQ.yml)

In [17]:
import os
model_to_test = 'hgq2'
model_revision = '2'
hls4ml_revision = 'VitisUnified_2025'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, model_revision)
os.makedirs(model_dir, exist_ok=True)

description = """
# Model Configuration

Aim is to run inference on HW (VitisUnified with custom 2025-script-patch)
Problems running HGQ2-models; Vitis Unified sets io_stream, but HGQ2 requires io_parallel for heteregenous activation. 
This is just to test different models.

- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os
from sklearn.metrics import accuracy_score

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)

In [19]:
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']
!vitis --version
!vitis_hls -version
!vivado -version


****** Vitis Development Environment
****** Vitis v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:14
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

/bin/bash: line 1: vitis_hls: command not found
vivado v2025.2 (64-bit)
Tool Version Limit: 2025.11
SW Build 6299465 on Fri Nov 14 12:34:56 MST 2025
IP Build 6300035 on Fri Nov 14 10:48:45 MST 2025
SharedData Build 6298862 on Thu Nov 13 04:50:51 MST 2025
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.


In [20]:
# Use absolute paths for data files
x_train_val_path = os.path.join(base_dir, "x_train_val.npy")
x_test_path = os.path.join(base_dir, "x_test.npy")
y_train_val_path = os.path.join(base_dir, "y_train_val.npy")
y_test_path = os.path.join(base_dir, "y_test.npy")
classes_path = os.path.join(base_dir, "classes.npy")

x_train_val = np.load(x_train_val_path)
x_test = np.load(x_test_path)
y_train_val = np.load(y_train_val_path)
y_test = np.load(y_test_path)

x_test.dtype

dtype('float32')

In [21]:
# Prepare subset of testdata for simulation (running everything takes an unnecessary long time)
simulation_rows = 100
x_test_sim_path = os.path.join(base_dir, "x_test_sim.npy")
y_test_sim_path = os.path.join(base_dir, "y_test_sim.npy")
np.save(x_test_sim_path, x_test[:simulation_rows])
np.save(y_test_sim_path, y_test[:simulation_rows])

In [22]:
keras_model_path = os.path.join(model_dir, f"model_HGQ.keras")

import hgq.layers
from keras.models import load_model
model = load_model(keras_model_path)

In [23]:
# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

# Convert and synthesize with HLS4ML
Configure parameters.
KV260: xck26-sfvc784-2LV-c 

In [24]:
import hls4ml

hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')

#hls_config['Model']['Strategy'] = 'Distributed Arithmetic'
proj_name = f"{str(model_to_test)}_{str(model_revision)}_hls4ml_prj_{str(hls4ml_revision)}"

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend='vitisunified',
    hls_config=hls_config,
    io_type='io_stream',
    proj_name = proj_name,
    output_dir=output_dir, 
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5',
    # Set input data for model simulation
    input_data_tb= x_test_sim_path,
    output_data_tb=y_test_sim_path,
)
hls_model.compile()
#hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True,to_file=os.path.join(output_dir, "model-plot.png"))

Check performance

In [25]:
y_keras = model.predict(x_test)
y_hls = hls_model.predict(np.ascontiguousarray(x_test))

print("Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):")
for x,y in enumerate(y_keras[:5]):
    print(f"{y-y_hls[x]}")

#print(y_keras[:10])
#print(y_hls[:10])
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))



5188/5188 ━━━━━━━━━━━━━━━━━━━━ 2s 447us/step
Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
Keras  Accuracy: 0.747132530120482
hls4ml Accuracy: 0.747132530120482


In [26]:
hls_model.build(
    #csim=False,
    synth=True,
    bitfile=True
)


****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Mon May  4 15:59:26 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/2/hls4ml_prj_VitisUnified_2025/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/2/hls4ml_prj_VitisUnified_2025/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/2/hls4ml_prj_VitisUnified_2025/vitis_workspace/myproject/vitis_unified_project'.
INFO: [HLS 200-1505] Using default flow_target 'vivado'
Resolution: For help on HLS 200-1505 see docs.amd.com/

# Simulation
[vitis unified tutorial](https://github.com/Tanawin1701d/vitis_unified_backend_tutorial/blob/master/03_co_simulation.ipynb)

In [27]:
# Build and do co-simulation
hls_model.build(
    #synth=True, # Only needs to run first time
    cosim=True,
    ) 

# Problem 1 19.03.2026
# Version missmatch mellom system glibc og Vitis 2023.2 (binutils)
# Kjøre i Ubuntu 22 docker, se over


****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Mon May  4 16:06:57 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/2/hls4ml_prj_VitisUnified_2025/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/2/hls4ml_prj_VitisUnified_2025/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/2/hls4ml_prj_VitisUnified_2025/vitis_workspace/myproject/vitis_unified_project'.
INFO: [HLS 200-1505] Using default flow_target 'vivado'
Resolution: For help on HLS 200-1505 see docs.amd.com/

In [28]:
y_baseline = hls_model.predict(np.ascontiguousarray(x_test[:simulation_rows]))
#y_baseline = model.predict(np.ascontiguousarray(x_test[:simulation_rows]))
y_simulation = np.loadtxt(os.path.join(output_dir, "tb_data/rtl_cosim_results.log"))

In [29]:
print(f"y_baseline shape: {y_baseline.shape} and y_simulation: {y_simulation.shape}")
#print(y_simulation)

y_baseline shape: (100, 5) and y_simulation: (100, 5)


In [30]:
assert np.allclose(y_baseline, y_simulation, rtol=0.0, atol=1e-4), (
    "The results from bridge and cosim are NOT equal!"
)
print("\n✅ RTL co-simulation comparison passed, absolute difference is less than 1e-4 (atol=1e-4).")


✅ RTL co-simulation comparison passed, absolute difference is less than 1e-4 (atol=1e-4).


In [31]:
from sklearn.metrics import accuracy_score
print("Difference in inference-calculations between HLS4ML bridge and simulated inference (first rows):")
abs_diff = y_baseline[:3] - y_simulation[:3]
print(np.round(abs_diff, 8))

mse = np.mean(np.square(y_baseline - y_simulation))
print(f"MSE: {mse}")


print("Baseline  Accuracy: {}".format(accuracy_score(np.argmax(y_test[:simulation_rows], axis=1), np.argmax(y_baseline, axis=1))))
print("Simulation Accuracy: {}".format(accuracy_score(np.argmax(y_test[:simulation_rows], axis=1), np.argmax(y_simulation, axis=1))))

Difference in inference-calculations between HLS4ML bridge and simulated inference (first rows):
[[-3.75e-06  3.75e-06  0.00e+00  0.00e+00  0.00e+00]
 [-1.25e-06 -3.75e-06  5.00e-06  0.00e+00  0.00e+00]
 [ 2.50e-07 -5.00e-07 -2.50e-06  0.00e+00  0.00e+00]]
MSE: 1.3737894999784499e-11
Baseline  Accuracy: 0.77
Simulation Accuracy: 0.77
